# 📊 04｜社區活動問卷：先清理重複與寫法，再看滿意度

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/04_survey_cleaning/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：原始問卷有**重複送出、同義不同寫法、不可能的值**；先訂清理規則、記錄每一步刪改了幾筆，才能計算滿意度。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：找出重複與亂寫的滿意度、訂好清理規則，最後用**一張垂直長條圖**回答「1～5 分各有多少人？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：清理其餘欄位（性別、居住區、年齡、電話），匯出乾淨的問卷檔與清理紀錄。
3. **選做挑戰（單元五～六）**：拆開多選欄位統計各活動參加人數，或比較各居住區的滿意度。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/survey_raw.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一份送出的問卷**（共 320 列，含重複送出）。清理後另存新檔，不得改寫原始檔。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`姓名`、`電話` ➔ 只用來辨識填答者；電話是文字，只統一格式，不做運算。
- 🔵 **條件 (Dimension)**：`性別`、`居住區`、`參加活動`（類別，寫法不一，需統一）；`填答時間`（時間，格式混雜，主線不用）。
- 🟢 **指標 (Metric)**：`滿意度`（1～5 的順序等級）、`年齡` ➔ 混有文字、不可能的值與空白。

### 2. 五大提問積木與核心問題
- 🎯 **分布（必做）**：清理後，滿意度 1～5 分各有多少人？哪個分數最多？
- 🔢 **比較（選做）**：各活動分別有多少人參加？哪個活動最多人？
- 🏘️ **比較（選做）**：各居住區的平均滿意度有差別嗎？每區有多少份有效問卷？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **先去重複再計數**：15 份是完全相同的重複送出；不刪除，分母就會多算。
- ⚠️ **同義寫法要統一，不可能的值不能硬改**：「5分」→5、「非常滿意」→5、「滿意」→4 是**我們訂的對照規則**，要寫清楚；「6」不在 1～5 範圍內，設為缺值，不能猜成 5。
- ⚠️ **有效樣本數要說清楚**：滿意度的分母是「有效填答數」，不是 320，也不是去重複後的全部份數。
- ⚠️ **人數 ≠ 人次**：多選題拆開後，各活動人數加起來會超過問卷份數，不能說成「總參加人數」。
- ⚠️ **滿意度是等級**：1～5 分是順序，不是等距的測量；平均只能當參考，並一定附上樣本數。

### 4. 決定最適圖表
- **1～5 分各有多少人** ➔ **垂直長條圖**（分數只有 5 個等級，依 1→5 排列，不可重新排序；Y 軸從 0 開始）。
- **各活動參加人數（選做）** ➔ **水平長條圖**（依人數排序）。
- **各居住區平均滿意度（選做）** ➔ **垂直長條圖**，並在柱子上標出有效樣本數 n。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'survey_raw.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/04_survey_cleaning/data/survey_raw.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料、檢查重複與各種寫法（貼標籤）

> 🎯 **思維重點**：先「看見」髒資料：重複列有幾筆？性別、居住區、滿意度各有幾種寫法？先檢查，不急著改。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/survey_raw.csv，所有欄位先以文字讀入：
1. 顯示前 8 筆資料、欄位名稱、總列數與各欄缺值數。
2. 計算有幾列是「整列完全重複」的資料。
3. 分別列出「性別」「居住區」「滿意度」每一種寫法出現的次數（包含空白）。
4. 用文字回報：哪些欄位是標識、條件、指標；滿意度欄位有哪些寫法不是 1～5 的數字？
先不要修改資料，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

raw = pd.read_csv('data/survey_raw.csv', dtype=str)   # 先全部當文字讀入，避免自動轉換藏起問題
print("--- 前 8 筆資料 ---")
display(raw.head(8))
print(f"\n總列數：{len(raw)}；完全重複的列：{raw.duplicated().sum()} 列")
print("\n各欄缺值數：", raw.isna().sum().to_dict())

for col in ['性別', '居住區', '滿意度']:
    print(f"\n【{col}】各種寫法與次數：")
    print(raw[col].value_counts(dropna=False).to_dict())

### 📝 單元二【必做】：邊界檢核：去重複、統一滿意度寫法、排除不可能的值

> 🎯 **思維重點**：依序處理：刪除完全重複 → 把同義寫法對應成 1～5 → 不在 1～5 的值設為缺值。每一步都印出筆數，最後得到「有效樣本數」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依下列規則清理，並在每一步印出處理前後筆數：
1. 刪除完全重複的列（保留第一筆），不要修改原始 CSV。
2. 統一滿意度寫法：去掉前後空白與「分」字；「非常滿意」對應 5、「滿意」對應 4。請把這個對照規則寫在註解中。
3. 轉成數字後，不在 1～5 範圍內的值（例如 6）設為缺值，不可以猜成其他分數；列出被設為缺值的原始寫法與筆數。
4. 回報：去重複後有幾份問卷？滿意度有效填答幾份？缺值幾份？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
survey = raw.drop_duplicates().copy()
print(f"① 刪除完全重複：{len(raw)} → {len(survey)} 份")

# 滿意度對照規則（教師與學生事先約定，報告中要寫明）：
#   「5分」→ 5（去掉「分」字）；「非常滿意」→ 5；「滿意」→ 4
text_map = {'非常滿意': '5', '滿意': '4'}
score_text = survey['滿意度'].str.strip().str.replace('分', '', regex=False).replace(text_map)
score = pd.to_numeric(score_text, errors='coerce')

invalid = score.notna() & ~score.between(1, 5)
print("② 被設為缺值的不可能值：", survey.loc[invalid, '滿意度'].value_counts().to_dict())
survey['滿意度_清理'] = score.where(~invalid)

print(f"③ 去重複後 {len(survey)} 份；滿意度有效 {survey['滿意度_清理'].notna().sum()} 份、"
      f"缺值 {survey['滿意度_清理'].isna().sum()} 份（原本空白＋不可能值）")

### 📝 單元三【必做】：提問積木【分布】— 滿意度 1～5 分各有多少人？（垂直長條圖）

> 🎯 **思維重點**：5 個有順序的等級看人數分布，用【垂直長條圖】；X 軸固定 1→5，不能依人數重新排序；標題寫出有效樣本數 n。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用清理後的滿意度，畫出 1～5 分人數的垂直長條圖：
1. 先產出次數表：分數 1～5、人數、占有效樣本的百分比（保留一位小數）。
2. X 軸為滿意度分數，固定依 1、2、3、4、5 排列（即使某分數是 0 人也要顯示）；Y 軸為人數，從 0 開始。
3. 在柱子上方標註人數與百分比。
4. 標題寫出有效樣本數 n，圖表文字全部使用繁體中文。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

counts = survey['滿意度_清理'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
n_valid = counts.sum()
score_table = pd.DataFrame({'人數': counts, '百分比(%)': (counts / n_valid * 100).round(1)})
score_table.index = score_table.index.astype(int).rename('滿意度')
display(score_table)

plt.figure(figsize=(8, 5))
bars = plt.bar([f'{s} 分' for s in score_table.index], score_table['人數'], color='#3498db', width=0.6)
for bar, pct in zip(bars, score_table['百分比(%)']):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 1.5, f'{h:.0f} 人\n({pct}%)', ha='center', va='bottom', fontsize=10)
plt.ylim(0, score_table['人數'].max() * 1.25)
plt.title(f'社區活動滿意度分布（有效樣本 n = {n_valid}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('滿意度（1 = 非常不滿意，5 = 非常滿意）', fontsize=12)
plt.ylabel('人數', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能指出原始資料的三類問題：重複送出、同義不同寫法、不可能的值，並說出各自的處理規則。
- [ ] 能展示每一步清理前後的筆數，說出滿意度的有效樣本數；長條圖與次數表一致，分數依 1→5 排列。
- [ ] 能用一句話解釋：為什麼「6 分」不能直接改成 5 分？
- 💬 **討論題**：如果把「滿意」也當成 5 分，結果會改變什麼？這個規則應該由誰決定？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 原始 320 列，刪除 15 筆完全重複後剩 **305 份**問卷。
- 滿意度統一寫法後，6 分與空白設為缺值，有效填答 **291 份**；**4 分最多（97 人）**，5 分 95 人。
- 長條圖依 1→5 排列、Y 軸從 0 開始，標題寫出有效樣本數。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：清理其餘欄位，匯出乾淨問卷與清理紀錄

> 🎯 **思維重點**：交付的不只是乾淨資料，還有**清理紀錄**：每個欄位用什麼規則、影響了幾筆，別人才能檢查。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請清理其餘欄位，並匯出乾淨問卷與清理紀錄：
1. 姓名去掉前後空白；性別統一為「男」「女」（M、male、男生 → 男；F、female、女生 → 女），空白保留為缺值。
2. 居住區去掉空白，缺「區」字的補上（例如「大安」→「大安區」）。
3. 年齡只取數字；不在 10～100 歲的值（例如 999）設為缺值。
4. 電話只保留數字，+886 開頭改為 0，9 碼且以 9 開頭的前面補 0；最後檢查是否都是 09 開頭的 10 碼。
5. 建立清理紀錄表（欄位、規則、受影響筆數），連同乾淨問卷匯出成 output/問卷清理結果.xlsx 的兩個工作表；匯出後重新讀取核對列數為 305。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import re

def count_changed(before, after):
    """原本有填、但清理後被改寫或設為缺值的筆數"""
    had_value = before.notna()
    if pd.api.types.is_numeric_dtype(after):
        before = pd.to_numeric(before, errors='coerce')
    return int((had_value & (before != after)).sum())

clean = survey.copy()
clean['姓名'] = survey['姓名'].str.strip()

gender_map = {'男': '男', 'M': '男', 'male': '男', '男生': '男', '女': '女', 'F': '女', 'female': '女', '女生': '女'}
clean['性別'] = survey['性別'].str.strip().map(gender_map)

area = survey['居住區'].str.strip()
clean['居住區'] = area.where(area.str.endswith('區'), area + '區')

age = pd.to_numeric(survey['年齡'].str.extract(r'(\d+)')[0], errors='coerce')
clean['年齡'] = age.where(age.between(10, 100))

def fix_phone(text):
    digits = re.sub(r'\D', '', str(text))       # 只留數字
    if digits.startswith('886'):                 # +886 912... → 0912...
        digits = '0' + digits[3:]
    if len(digits) == 9 and digits.startswith('9'):  # 912345678 → 0912345678
        digits = '0' + digits
    return digits

clean['電話'] = survey['電話'].map(fix_phone)
print('電話全部為 09 開頭 10 碼：', clean['電話'].str.fullmatch(r'09\d{8}').all())

clean['滿意度'] = clean.pop('滿意度_清理')
log_df = pd.DataFrame([
    ('全部', '刪除完全重複的列', len(raw) - len(survey)),
    ('滿意度', '同義寫法對應 1～5；不在 1～5 設為缺值', count_changed(survey['滿意度'], clean['滿意度'])),
    ('姓名', '去除前後空白', count_changed(survey['姓名'], clean['姓名'])),
    ('性別', '多種寫法統一為男／女', count_changed(survey['性別'], clean['性別'])),
    ('居住區', '去空白、補上「區」字', count_changed(survey['居住區'], clean['居住區'])),
    ('年齡', '只取數字；不在 10～100 歲設為缺值', count_changed(survey['年齡'], clean['年齡'])),
    ('電話', '只留數字、+886 改 0、補前導 0', count_changed(survey['電話'], clean['電話'])),
], columns=['欄位', '清理規則', '受影響筆數'])
display(log_df)

os.makedirs('output', exist_ok=True)
output_path = 'output/問卷清理結果.xlsx'
with pd.ExcelWriter(output_path) as writer:
    clean.to_excel(writer, sheet_name='乾淨問卷', index=False)
    log_df.to_excel(writer, sheet_name='清理紀錄', index=False)
print('重新讀取乾淨問卷列數：', len(pd.read_excel(output_path, sheet_name='乾淨問卷')))

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 檔案！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較】— 各活動有多少人參加？（水平長條圖）

> 🎯 **思維重點**：多選題一格有多個答案，要先拆開再計數；拆開後的加總是「人次」，會大於問卷份數。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請統計「參加活動」這個多選欄位：
1. 使用去重複後的 305 份問卷，把分隔符號「;」「、」「,」都視為分隔，拆成一列一個活動，並去掉空白。
2. 計算每個活動的參加人數，畫出水平長條圖（依人數排序，數值軸從 0 開始，標註人數）。
3. 印出：問卷份數、各活動人數加總（人次），並說明兩者為什麼不同。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
activities = (survey['參加活動'].str.split(r'[;、,]')
              .explode().str.strip().loc[lambda s: s.ne('') & s.notna()])
activity_counts = activities.value_counts().sort_values()
print(f"問卷份數：{len(survey)}；各活動人數加總（人次）：{activity_counts.sum()}")
print("💡 一個人可以參加多個活動，所以人次會大於問卷份數，不能說成「總參加人數」。")

plt.figure(figsize=(8, 4.5))
bars = plt.barh(activity_counts.index, activity_counts.values, color='#3498db')
for bar in bars:
    plt.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, f'{bar.get_width():.0f} 人', va='center')
plt.xlim(0, activity_counts.max() * 1.15)
plt.title(f'各活動參加人數（問卷 n = {len(survey)}，可複選）', fontsize=13, fontweight='bold')
plt.xlabel('參加人數（人）')
plt.show()

### 📝 單元六【選做挑戰】：提問積木【比較】— 各居住區的平均滿意度（垂直長條圖＋樣本數）

> 🎯 **思維重點**：居住區要先統一寫法才能分組；每區樣本數不同，平均值旁一定要標 n，差距很小時不要誇大。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請使用單元四清理後的資料，比較各居住區的滿意度：
1. 依居住區計算有效滿意度的份數 n 與平均分數（保留兩位小數）。
2. 畫出垂直長條圖，Y 軸從 0 開始、上限 5，柱子上標示平均分與 n。
3. 用文字說明：最高與最低的區差距多少？在樣本數這麼少、滿意度又是等級資料的情況下，結論要怎麼保守描述？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
area_score = (clean.dropna(subset=['滿意度']).groupby('居住區')['滿意度']
              .agg(n='count', 平均滿意度='mean').round(2).sort_values('平均滿意度', ascending=False))
display(area_score)

plt.figure(figsize=(8, 4.5))
bars = plt.bar(area_score.index, area_score['平均滿意度'], color='#3498db', width=0.55)
for bar, n in zip(bars, area_score['n']):
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.05, f'{h:.2f}\n(n={n})', ha='center', va='bottom', fontsize=10)
plt.ylim(0, 5.5)
plt.title('各居住區平均滿意度（1～5 分）', fontsize=13, fontweight='bold')
plt.ylabel('平均滿意度（分）')
plt.show()
gap = area_score['平均滿意度'].max() - area_score['平均滿意度'].min()
print(f"💡 最高與最低相差 {gap:.2f} 分；每區只有約 40～60 份，差距不大時只能說「看不出明顯差異」。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：320 列去重複後 305 份；滿意度有效 291 份，長條圖依 1→5 排列並標示 n。
- [ ] **報表應用**：清理紀錄寫明每個欄位的規則與筆數；電話全部為 09 開頭 10 碼。
- [ ] **選做挑戰**：多選題拆開後能區分人數與人次；居住區比較附上樣本數、不誇大差距。
- [ ] **AI 協作**：能說明自己訂的清理規則、AI 幫忙執行的事，以及自己如何確認結果。